# 🧠 Entrenamiento Red Neuronal — Detector de Spam en YouTube

**Proyecto I: Introducción a la IA** — Universitat Politècnica de València

**Equipo:** Ángel del Campo · Samuel Litago · Artur Mas · Pablo

---

Este notebook implementa la **Red Neuronal Superficial** (Sprint 3.2) para clasificar comentarios de YouTube como Spam / No Spam.

**Arquitectura del modelo:**
- Entrada: TF-IDF (5 000 features) + 5 features EDA = 5 005 dimensiones
- Capa oculta 1: Dense(256, ReLU) + BatchNorm + Dropout(0.3)
- Capa oculta 2: Dense(128, ReLU) + BatchNorm + Dropout(0.2)
- Capa oculta 3: Dense(64, ReLU)
- Salida: Dense(1, Sigmoid)

**Entrenamiento:** Adam + EarlyStopping + ReduceLROnPlateau

## 1. Instalación de dependencias

In [ ]:
# Ejecutar solo si no están instaladas
!pip install tensorflow scikit-learn pandas numpy matplotlib seaborn joblib wordcloud -q
print('✅ Dependencias instaladas correctamente')

## 2. Importaciones

In [ ]:
import os
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Estilo visual
plt.style.use('dark_background')
COLORS = {'spam': '#ff4455', 'nospam': '#00ff88', 'accent': '#00d4ff', 'purple': '#a855f7'}

print(f'✅ TensorFlow {tf.__version__}')
print(f'✅ GPU disponible: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 3. Carga y unificación de datos

In [ ]:
# ── Si estás en Colab, sube los CSV con este bloque ──────────────────────────
# from google.colab import files
# uploaded = files.upload()
# ─────────────────────────────────────────────────────────────────────────────

DATA_DIR = '.'  # Ajustar si los CSV están en otra carpeta

dfs = []

# Dataset 1: YouTube Spam Dataset original (1 956 filas)
path1 = os.path.join(DATA_DIR, 'Youtube-Spam-Dataset.csv')
if os.path.exists(path1):
    df1 = pd.read_csv(path1)
    df1 = df1[['CONTENT', 'CLASS']].rename(columns={'CONTENT': 'text', 'CLASS': 'label'})
    df1['fuente'] = 'youtube_spam_original'
    dfs.append(df1)
    print(f'Dataset 1: {len(df1):,} filas  |  spam={df1["label"].sum():,}')

# Dataset 2: YouTube 45K (45 005 filas)
path2 = os.path.join(DATA_DIR, 'YouTube Comments Dataset with Sentiment Toxicity and Spam Labels (45K Rows).csv')
if os.path.exists(path2):
    df2 = pd.read_csv(path2)
    df2 = df2[['comment_text', 'label_spam']].dropna()
    df2 = df2.rename(columns={'comment_text': 'text', 'label_spam': 'label'})
    df2['label'] = pd.to_numeric(df2['label'], errors='coerce').fillna(0).astype(int)
    df2['fuente'] = 'youtube_45k'
    dfs.append(df2)
    print(f'Dataset 2: {len(df2):,} filas  |  spam={df2["label"].sum():,}')

if not dfs:
    raise FileNotFoundError('No se encontró ningún CSV. Sube los archivos al entorno.')

df = pd.concat(dfs, ignore_index=True)
df = df.dropna(subset=['text', 'label'])
df['text'] = df['text'].astype(str).str.strip()
df['label'] = df['label'].astype(int)
df = df[df['text'].str.len() > 2].reset_index(drop=True)

print(f'\n🔵 Dataset unificado: {len(df):,} comentarios')
print(df['label'].value_counts().rename({0: 'No Spam', 1: 'Spam'}))

## 4. Ingeniería de features (Sprint 2 — variables EDA)

In [ ]:
SPAM_WORDS = re.compile(
    r'subscribe|suscrib|check.?out|my.?channel|free|gratis|click|win|gana|'
    r'giveaway|sorteo|crypto|bitcoin|investment|earn|profit|dm.?me|'
    r'followers|views|likes|watch.?now|promo|discount|descuento|link',
    re.I
)

def engineer_features(texts: pd.Series) -> np.ndarray:
    """Extrae 5 features numéricas derivadas del análisis EDA (Sprint 2)."""
    t = texts.astype(str)
    n_chars   = t.str.len().fillna(0)
    ratio_cap = (t.str.count(r'[A-Z]') / n_chars.clip(lower=1)).fillna(0)
    has_url   = t.str.contains(r'https?://|www\.|bit\.ly|\.com/', case=False, na=False).astype(int)
    log_excl  = np.log1p(t.str.count(r'!').fillna(0))
    spam_wds  = t.apply(lambda x: 1 if SPAM_WORDS.search(x) else 0)
    return np.column_stack([n_chars, ratio_cap, has_url, log_excl, spam_wds])

FEATURE_NAMES = ['longitud_chars', 'ratio_mayusculas', 'contiene_url',
                 'log_exclamaciones', 'palabras_spam']

X_eng = engineer_features(df['text'])
print(f'Features EDA generadas: {X_eng.shape}')
print(pd.DataFrame(X_eng, columns=FEATURE_NAMES).describe().round(3))

## 5. Vectorización TF-IDF + división train/val/test

In [ ]:
# División estratificada: 70% train — 15% val — 15% test
idx_train, idx_temp = train_test_split(
    df.index, test_size=0.30, random_state=42, stratify=df['label']
)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, random_state=42, stratify=df.loc[idx_temp, 'label']
)

print(f'Train: {len(idx_train):,} | Val: {len(idx_val):,} | Test: {len(idx_test):,}')

# TF-IDF entrenado SOLO sobre train (evitar data leakage)
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='word',
    min_df=2
)
tfidf.fit(df.loc[idx_train, 'text'])

def build_X(idx):
    X_tf = tfidf.transform(df.loc[idx, 'text']).toarray().astype(np.float32)
    X_e  = engineer_features(df.loc[idx, 'text']).astype(np.float32)
    # Normalizar features EDA con stats de train
    return np.hstack([X_tf, X_e])

X_train = build_X(idx_train);  y_train = df.loc[idx_train, 'label'].values
X_val   = build_X(idx_val);    y_val   = df.loc[idx_val,   'label'].values
X_test  = build_X(idx_test);   y_test  = df.loc[idx_test,  'label'].values

print(f'\nDimensión de entrada al modelo: {X_train.shape[1]} features')
print(f'  → TF-IDF: 5 000  |  EDA: 5')

## 6. Arquitectura de la Red Neuronal Superficial

In [ ]:
INPUT_DIM = X_train.shape[1]

def build_model(input_dim: int) -> keras.Model:
    """
    Red Neuronal Superficial (Shallow Neural Network) para clasificación binaria.
    
    Arquitectura justificada en Sprint 3.1:
    - 3 capas ocultas (Red 'superficial' vs deep learning con >10 capas)
    - BatchNormalization: estabiliza distribuciones intermedias
    - Dropout: regularización para evitar overfitting
    - Sigmoid final: salida como probabilidad [0, 1]
    """
    inp = keras.Input(shape=(input_dim,), name='input')

    x = layers.Dense(256, kernel_regularizer=regularizers.l2(1e-4), name='dense_1')(inp)
    x = layers.BatchNormalization(name='bn_1')(x)
    x = layers.Activation('relu', name='relu_1')(x)
    x = layers.Dropout(0.30, name='drop_1')(x)

    x = layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4), name='dense_2')(x)
    x = layers.BatchNormalization(name='bn_2')(x)
    x = layers.Activation('relu', name='relu_2')(x)
    x = layers.Dropout(0.20, name='drop_2')(x)

    x = layers.Dense(64, activation='relu', name='dense_3')(x)

    out = layers.Dense(1, activation='sigmoid', name='output')(x)

    model = keras.Model(inputs=inp, outputs=out, name='SpamDetector_v1')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
            keras.metrics.AUC(name='auc')
        ]
    )
    return model

model = build_model(INPUT_DIM)
model.summary()

## 7. Entrenamiento con Early Stopping

In [ ]:
os.makedirs('model', exist_ok=True)

callbacks = [
    EarlyStopping(
        monitor='val_loss', patience=7,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-6, verbose=1
    ),
    ModelCheckpoint(
        'model/spam_model_best.keras',
        monitor='val_auc', save_best_only=True,
        mode='max', verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=128,
    callbacks=callbacks,
    verbose=1
)

print(f'\n✅ Entrenamiento completado en {len(history.history["loss"])} épocas')

## 8. Curvas de entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0d0d1a')
for ax in axes:
    ax.set_facecolor('#1a1a2e')
    ax.grid(True, alpha=0.2, color='#94a3b8')

epocas = range(1, len(history.history['loss']) + 1)

# Loss
axes[0].plot(epocas, history.history['loss'],     color=COLORS['spam'],   lw=2, label='Train')
axes[0].plot(epocas, history.history['val_loss'], color=COLORS['accent'], lw=2, label='Validación', ls='--')
axes[0].set_title('Loss (Binary Crossentropy)', color='white', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].tick_params(colors='#94a3b8')

# Accuracy
axes[1].plot(epocas, history.history['accuracy'],     color=COLORS['nospam'], lw=2, label='Train')
axes[1].plot(epocas, history.history['val_accuracy'], color=COLORS['purple'], lw=2, label='Validación', ls='--')
axes[1].set_title('Accuracy', color='white', fontsize=13, fontweight='bold')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
axes[1].legend()
axes[1].tick_params(colors='#94a3b8')

# AUC
axes[2].plot(epocas, history.history['auc'],     color=COLORS['accent'], lw=2, label='Train')
axes[2].plot(epocas, history.history['val_auc'], color=COLORS['spam'],   lw=2, label='Validación', ls='--')
axes[2].set_title('AUC-ROC', color='white', fontsize=13, fontweight='bold')
axes[2].legend()
axes[2].tick_params(colors='#94a3b8')

fig.suptitle('Curvas de Entrenamiento — Red Neuronal Spam Detector', color='white', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('curvas_entrenamiento.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()
print('📊 Guardado: curvas_entrenamiento.png')

## 9. Evaluación en conjunto de test

In [ ]:
y_prob = model.predict(X_test, verbose=0).flatten()
y_pred = (y_prob >= 0.50).astype(int)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)

print('=' * 60)
print('   RESULTADOS EN EL CONJUNTO DE TEST')
print('=' * 60)
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'  AUC-ROC   : {auc:.4f}')
print('=' * 60)
print()
print(classification_report(y_test, y_pred, target_names=['No Spam', 'Spam']))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d0d1a')

# Heatmap
ax = axes[0]
ax.set_facecolor('#1a1a2e')
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['No Spam', 'Spam'],
    yticklabels=['No Spam', 'Spam'],
    ax=ax, linewidths=0.5, linecolor='#0d0d1a'
)
ax.set_title('Matriz de Confusión', color='white', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicción', color='#94a3b8')
ax.set_ylabel('Real', color='#94a3b8')
ax.tick_params(colors='#94a3b8')

# ROC Curve
ax2 = axes[1]
ax2.set_facecolor('#1a1a2e')
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax2.plot(fpr, tpr, color=COLORS['accent'], lw=2, label=f'ROC (AUC = {auc:.3f})')
ax2.plot([0, 1], [0, 1], color='#94a3b8', lw=1, ls='--', label='Azar')
ax2.fill_between(fpr, tpr, alpha=0.1, color=COLORS['accent'])
ax2.set_xlabel('Tasa de Falsos Positivos', color='#94a3b8')
ax2.set_ylabel('Tasa de Verdaderos Positivos', color='#94a3b8')
ax2.set_title('Curva ROC', color='white', fontsize=14, fontweight='bold')
ax2.legend(loc='lower right', facecolor='#1a1a2e', labelcolor='white')
ax2.tick_params(colors='#94a3b8')
ax2.grid(True, alpha=0.2, color='#94a3b8')

plt.tight_layout()
plt.savefig('matriz_confusion_roc.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()
print('📊 Guardado: matriz_confusion_roc.png')

## 10. Guardar modelo y vectorizador

In [ ]:
os.makedirs('model', exist_ok=True)

# Guardar modelo Keras
model.save('model/spam_model.keras')
print('✅ Modelo guardado: model/spam_model.keras')

# Guardar también en formato .h5
model.save('model/spam_model.h5')
print('✅ Modelo guardado: model/spam_model.h5')

# Guardar vectorizador TF-IDF
joblib.dump(tfidf, 'model/tfidf_vectorizer.pkl')
print('✅ Vectorizador guardado: model/tfidf_vectorizer.pkl')

# Guardar métricas finales
import json
metrics_dict = {
    'accuracy': float(acc), 'precision': float(prec),
    'recall': float(rec),   'f1': float(f1), 'auc_roc': float(auc),
    'model_type': 'Keras MLP (Red Neuronal Superficial)',
    'input_dim': int(INPUT_DIM), 'architecture': '256-128-64-1',
    'train_size': int(len(idx_train)), 'test_size': int(len(idx_test))
}
with open('model/metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=2)
print('✅ Métricas guardadas: model/metrics.json')

print('\n' + '='*50)
print('  RESUMEN FINAL')
print('='*50)
print(f'  Accuracy : {acc*100:.2f}%')
print(f'  F1-Score : {f1:.4f}')
print(f'  AUC-ROC  : {auc:.4f}')
print('='*50)
print('\n👉 Descarga la carpeta model/ para usarla en la app Streamlit')
print('   O sube los archivos al repositorio GitHub')

## 11. Test rápido del modelo guardado

Verificación de que el modelo se carga y predice correctamente.

In [ ]:
# Cargar y verificar
loaded_model = keras.models.load_model('model/spam_model.keras')
loaded_tfidf = joblib.load('model/tfidf_vectorizer.pkl')

ejemplos = [
    ('Subscribe to my channel! FREE music! Click here: http://bit.ly/xxx', 1),
    ('This song is amazing, I listen to it every day', 0),
    ('WIN FREE CRYPTO! Subscribe NOW and earn money! GIVEAWAY!!!', 1),
    ('I love this band, been a fan since 2008', 0),
]

print('\n🧪 Test con ejemplos conocidos:\n')
for texto, etiq_real in ejemplos:
    X_tf  = loaded_tfidf.transform([texto]).toarray().astype(np.float32)
    X_e   = engineer_features(pd.Series([texto])).astype(np.float32)
    X_inp = np.hstack([X_tf, X_e])
    prob  = float(loaded_model.predict(X_inp, verbose=0)[0][0])
    pred  = '🚨 SPAM' if prob >= 0.5 else '✅ No Spam'
    real  = '🚨 SPAM' if etiq_real == 1 else '✅ No Spam'
    ok    = '✔' if (prob >= 0.5) == etiq_real else '✘'
    print(f'  [{ok}] {pred} ({prob:.1%})  |  Real: {real}')
    print(f'      "{texto[:70]}..."\n' if len(texto) > 70 else f'      "{texto}"\n')